# Support Vector Machines: From First Principles

## 1. Why Look for a Wide Margin?

Imagine classifying manufactured parts as acceptable or defective using two standardized measurements. Several straight lines might separate the training classes perfectly. Which line should we choose? A line passing very close to the nearest observations could change its predictions after a small measurement disturbance. A line with more space on both sides may provide a more stable separation.

A Support Vector Machine (SVM) formalizes this preference with a **margin**. For perfectly separable data, it finds the separating boundary with the largest minimum distance to the training points. For overlapping data, it balances a wide margin against violations of that separation.

<div style="color:#17324d; background-color:#eaf4ff; border-left:5px solid #2878b5; padding:10px 14px; margin:12px 0;"><strong>Core intuition</strong><br>Choose a boundary that separates classes with room to spare, while allowing some violations when insisting on perfect separation would be unreasonable.</div>

A margin is a geometric quantity in the chosen feature representation. It is not a probability or a guarantee of correct predictions. We will start with a linear binary classifier, calculate a boundary by hand, then introduce soft margins and nonlinear kernels.

## 2. Inputs, Outputs, and the Decision Boundary

Support Vector Classification is supervised learning: training rows contain features and known class labels. A fitted model receives a new feature vector and returns a class prediction. SVM methods also include regression variants, but this lesson focuses on classification to match the companion.

For binary mathematics, encode the labels as $y_i\in\{-1,+1\}$. With $p$ features, a linear SVM computes

$$
f(\mathbf x)=\mathbf w^T\mathbf x+b=\sum_{j=1}^{p}w_jx_j+b.
$$

| Symbol or term | Meaning |
|---|---|
| $\mathbf x$ | Feature vector for one observation |
| $\mathbf w$ | Learned coefficients, one per feature |
| $b$ | Learned intercept |
| $f(\mathbf x)$ | Signed decision score |
| Decision boundary | Points satisfying $f(\mathbf x)=0$ |
| Hyperplane | A flat boundary: a line in two dimensions, a plane in three |
| Support vector | Training row with a nonzero weight in the fitted dual representation |

Predict $+1$ when the score is positive and $-1$ when it is negative. A score of exactly zero requires an implementation tie convention. The coefficients orient the boundary, and the intercept positions it. Libraries can accept other label encodings and map them internally; the $-1,+1$ notation is for deriving the objective.

A large score magnitude indicates separation from the boundary within this fitted model, not a calibrated chance of correctness.

## 3. Scores, Distances, and the Margin

The Euclidean length of the coefficient vector is

$$
\|\mathbf w\|=\sqrt{\sum_jw_j^2}.
$$

For nonzero $\mathbf w$, the perpendicular distance from a point to the boundary is

$$
\frac{|\mathbf w^T\mathbf x+b|}{\|\mathbf w\|}.
$$

Dividing by the norm matters. Multiplying both $\mathbf w$ and $b$ by 10 changes every score by a factor of 10 but leaves the boundary and geometric distances unchanged. We therefore need a consistent normalization before discussing coefficient size and margins.

For separable data, use the constraints $y_if(\mathbf x_i)\geq1$. The two margin planes are $f(\mathbf x)=+1$ and $f(\mathbf x)=-1$. The distance from the boundary to either plane is $1/\|\mathbf w\|$, and the full width between them is

$$
\mathrm{margin\ width}=\frac{2}{\|\mathbf w\|}.
$$

Some explanations call the one-sided distance “the margin”; others use the full width. Here we state which is intended. Smaller coefficient norm means a wider margin only under the normalized constraints, not after arbitrary score rescaling.

## 4. Hard-Margin Optimization

For $n$ linearly separable training rows, the hard-margin problem is

$$
\min_{\mathbf w,b}\frac12\|\mathbf w\|^2
\quad\text{subject to}\quad
 y_i(\mathbf w^T\mathbf x_i+b)\geq1\quad\text{for every }i.
$$

The constraints require all rows to lie on the correct side and at least as far as their class's margin plane. Minimizing the squared norm maximizes the geometric margin; the factor one half simplifies derivatives. The intercept is not penalized in this formulation.

The signed quantity $y_if(\mathbf x_i)$ is called the functional margin. Positive values indicate the correct side, but the constraint requires at least 1. Thus “correct classification” and “satisfying the margin requirement” are different conditions.

If the classes overlap in a way no hyperplane can separate, this problem has no feasible solution. Even when separation is possible, one unusual observation can force a very narrow margin. These limitations motivate a soft margin rather than an insistence on perfect training separation.

## 5. Worked Example: Find the Maximum-Margin Boundary

Consider one standardized measurement with negative-class values 1 and 2, and positive-class values 4 and 5. A separating threshold can be placed anywhere strictly between 2 and 4, but the midpoint 3 gives the greatest clearance from the closest opposite-class points.

Write the score as $f(x)=wx+b$, with $w>0$. At the maximum-margin solution, the closest points satisfy

$$
2w+b=-1,\qquad4w+b=1.
$$

Subtracting gives $2w=2$, so $w=1$ and $b=-3$. Therefore $f(x)=x-3$, the decision boundary is $x=3$, and the full margin width is $2/|w|=2$.

| Input $x$ | Label $y$ | Score $f(x)$ | Signed margin $yf(x)$ |
|---|---:|---:|---:|
| 1 | -1 | -2 | 2 |
| 2 | -1 | -1 | 1 |
| 4 | +1 | 1 | 1 |
| 5 | +1 | 2 | 2 |

The rows at 2 and 4 support the two margin planes. The rows at 1 and 5 lie farther away. To verify optimality, the constraints at 2 and 4 require $2w+b\leq-1$ and $4w+b\geq1$, hence $w\geq1$. Our solution achieves the smallest feasible norm.

For a new input 3.5, the score is 0.5, so the prediction is positive. Its geometric distance to the boundary is 0.5. It lies inside the positive margin, showing that a correctly classified point can still violate the margin requirement.

<div style="color:#17324d; background-color:#eaf4ff; border-left:5px solid #2878b5; padding:10px 14px; margin:12px 0;"><strong>Worked-example takeaway</strong><br>The support points determine the clearance in this simple fit. A score of 0.5 describes a position relative to the boundary, not a 50% class probability.</div>

## 6. Soft Margins and Slack Variables

Real classes often overlap. Introduce a nonnegative slack variable $\xi_i$ for each row to measure the allowed margin shortfall:

$$
\min_{\mathbf w,b,\boldsymbol\xi}\frac12\|\mathbf w\|^2+C\sum_{i=1}^{n}\xi_i,
$$

$$
y_i(\mathbf w^T\mathbf x_i+b)\geq1-\xi_i,\qquad \xi_i\geq0.
$$

Here $C>0$ controls the cost of violations, and $\boldsymbol\xi$ collects the slack values. At an optimum, the smallest feasible slack is $\xi_i=\max(0,1-y_if(\mathbf x_i))$.

| Signed margin $yf(x)$ | Smallest slack | Interpretation |
|---|---|---|
| At least 1 | 0 | Correct side, on or outside the margin |
| Between 0 and 1 | Between 0 and 1 | Correct side, inside the margin |
| 0 | 1 | On the decision boundary |
| Negative | Greater than 1 | Wrong side of the boundary |

Slack does not simply count mistakes. It also penalizes correct predictions that are too close to the boundary, and larger wrong-side scores incur larger penalties.

Support vectors in a soft-margin fit can lie on the margin, inside it, or on the wrong side of the boundary. Calling them only “the closest correctly classified points” would omit important cases.

## 7. Hinge Loss and the Meaning of C

Substituting the smallest feasible slack into the objective gives

$$
J(\mathbf w,b)=\frac12\|\mathbf w\|^2+C\sum_i\max(0,1-y_if(\mathbf x_i)).
$$

The expression $\max(0,1-yf)$ is **hinge loss**. For signed margins of 1.5, 0.4, 0, and -0.6, the losses are respectively 0, 0.6, 1, and 1.6. Loss stops decreasing once a point is correctly beyond the margin.

A smaller $C$ gives more relative emphasis to a small coefficient norm and tolerates more margin shortfall. A larger $C$ makes violations more expensive and may produce a tighter, more data-sensitive boundary. Neither direction guarantees better test performance. $C$ is not a probability threshold and not the permitted number of mistakes.

To compare objective conventions, divide the displayed summed-loss objective by $Cn$:

$$
\frac{1}{2Cn}\|\mathbf w\|^2+\frac1n\sum_i\max(0,1-y_if(\mathbf x_i)).
$$

If another text writes an averaged hinge loss plus $(\lambda/2)\|\mathbf w\|^2$, the matching value is $\lambda=1/(Cn)$ for this dataset. This explains why increasing $C$ corresponds to weaker relative regularization, unlike increasing Ridge's penalty alpha.

<div style="color:#664b12; background-color:#fff5df; border-left:5px solid #b7791f; padding:10px 14px; margin:12px 0;"><strong>Common pitfall</strong><br>A large C asks the optimizer to reduce margin violations more strongly. It does not guarantee zero training errors, good generalization, or immunity to mislabeled rows.</div>

## 8. Support Vectors in the Prediction Rule

An alternative, equivalent optimization formulation assigns nonnegative weights $\alpha_i$ to training rows. For the ordinary unweighted binary soft-margin problem, they satisfy $0\leq\alpha_i\leq C$. The linear coefficient vector can be expressed as

$$
\mathbf w=\sum_{i=1}^{n}\alpha_i y_i\mathbf x_i.
$$

Rows with nonzero $\alpha_i$ are support vectors. Substitution gives

$$
f(\mathbf x)=\sum_{i:\alpha_i>0}\alpha_i y_i\mathbf x_i^T\mathbf x+b.
$$

The dot product $\mathbf x_i^T\mathbf x$ multiplies corresponding coordinates and adds them. Prediction combines comparisons with the support vectors rather than assigning every training row a nonzero contribution.

These $\alpha_i$ are fitted optimization weights, not the regularization hyperparameter called alpha in other algorithms. The bounds can change when samples or classes receive different penalties.

A row strictly outside its correct margin has zero weight in the ideal solution. Rows with $0<\alpha_i<C$ lie on a margin plane; rows at the upper bound may be inside the margin or misclassified. Degenerate configurations and numerical tolerance mean that merely sitting on a margin does not guarantee a strictly positive weight.

Non-support rows still participated in fitting. Their absence from the final sum does not mean their labels or positions could be changed arbitrarily without affecting a refit.

## 9. Why We Need Kernels

A straight boundary cannot describe every classification pattern. Suppose values near zero belong to the negative class while large positive and negative values belong to the positive class. For training inputs $-2,-1,1,2$, assign positive labels to $-2,2$ and negative labels to $-1,1$. No single threshold on $x$ separates them.

Transform the input to $z=x^2$. The transformed values are 4, 1, 1, 4. A boundary at $z=2.5$ now separates the classes. In the original input space, the decision rule is based on $x^2-2.5$, creating two boundary locations at $x=\pm\sqrt{2.5}$.

This example shows how a linear rule in transformed features can be nonlinear in the original input. A **kernel** computes a dot product in such a feature space:

$$
K(\mathbf x,\mathbf z)=\phi(\mathbf x)^T\phi(\mathbf z),
$$

where $\phi$ is the feature transformation. Replace dot products in the support-vector prediction rule to obtain

$$
f(\mathbf x)=\sum_{i:\alpha_i>0}\alpha_i y_iK(\mathbf x_i,\mathbf x)+b.
$$

The kernel trick can avoid constructing the transformed vectors explicitly. The optimization still learns weights and an intercept; the kernel is not itself a classifier. Standard valid kernels provide the inner-product structure required by the usual convex formulation. An arbitrary similarity formula need not do so.

## 10. Linear, Polynomial, and RBF Kernels

| Kernel | Formula | Main interpretation |
|---|---|---|
| Linear | $K(\mathbf x,\mathbf z)=\mathbf x^T\mathbf z$ | A flat boundary in supplied features |
| Polynomial | $K(\mathbf x,\mathbf z)=(\gamma\mathbf x^T\mathbf z+r)^d$ | Powers and interactions controlled by degree and scaling |
| RBF | $K(\mathbf x,\mathbf z)=\exp(-\gamma\lVert\mathbf x-\mathbf z\rVert^2)$ | Similarity decreases with squared distance |

Here $d$ is the polynomial degree, $r$ is the polynomial offset called `coef0`, and $\gamma$ is a kernel scale parameter. RBF means radial basis function. The same symbol gamma appears in different kernel formulas, so its effect should be understood in context.

For RBF, if squared distance is 4, gamma 0.1 gives similarity $e^{-0.4}\approx0.6703$, while gamma 1 gives $e^{-4}\approx0.0183$. A larger gamma makes the influence of a point more local. A smaller gamma gives broader similarity and often a smoother boundary.

High gamma together with a large C can allow highly localized fitting and overfitting. A very small gamma can fail to represent useful structure, but C interacts with that choice. Tune them jointly rather than treating one as an independent complexity knob.

RBF similarity is not a class probability. It compares two feature vectors and does not include labels. Nor does a high-dimensional mapping guarantee useful separation on future data.

## 11. Scaling and Feature Representation

Margins, dot products, and RBF distances depend on feature units. If one measurement is in thousands and another in fractions, the first can dominate geometric comparisons. This changes both what the kernel considers similar and how coefficient size is penalized.

A common transformation is standardization:

$$
z_{ij}=\frac{x_{ij}-\mu_j}{s_j},
$$

where $\mu_j$ and $s_j$ are feature $j$'s training mean and standard deviation. Fit these statistics on training rows only and reuse them for validation and future inputs. Scaling makes numerical units comparable; it does not guarantee that the features define a useful geometry.

Handle missing values explicitly, encode categorical features appropriately, and investigate extreme observations. SVMs can be sensitive to outliers, especially when C strongly penalizes their violations. Sparse text often needs a representation and normalization designed for sparse data; mean-centering a sparse matrix can destroy its memory advantage.

In scikit-learn SVC, `gamma="scale"` uses $1/(p\operatorname{Var}(X))$, based on the data supplied to the estimator. With standardized, nonconstant dense columns this is approximately $1/p$. Constant columns and preprocessing choices matter. This is an implementation default, not a guarantee of an optimal kernel width. See the [SVC parameter reference](https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVC.html).

<div style="color:#22543d; background-color:#edf8f1; border-left:5px solid #2f855a; padding:10px 14px; margin:12px 0;"><strong>Practical tip</strong><br>Keep preprocessing and SVC in one pipeline, then cross-validate the pipeline. Each fold must learn its scaling from its own training rows.</div>

## 12. Training, Tuning, and Using the Model

1. Define the labels, available prediction-time inputs, and which errors matter most. Set aside a final test set using appropriate stratification, time ordering, or grouping.
2. Build a preprocessing-and-classifier pipeline. Start with a linear baseline to assess whether nonlinear flexibility is needed.
3. Use training-only cross-validation to compare kernels and logarithmically spaced C values. For RBF, compare combinations of C and gamma.
4. Inspect per-class metrics and variability across folds. Refine a search range if useful candidates lie at its edge; do not assume a fixed grid suits every dataset.
5. Select the configuration, refit on all development training data, and evaluate once on the reserved test set.
6. Apply the fitted pipeline to new raw inputs and monitor performance as data distributions change.

With the companion's pipeline names, tune `model__C`, `model__kernel`, and, where relevant, `model__gamma`. Example positive numeric candidates such as $0.01,0.1,1,10$ provide a search starting point, not universal settings. Gamma has no role in the linear kernel.

| Choice | What it controls | Caution |
|---|---|---|
| C | Cost of margin violations relative to coefficient norm | Larger is not automatically better |
| Kernel | Geometry available to the classifier | Added flexibility needs validation |
| Gamma | RBF locality or polynomial scaling | Interacts with C and feature scaling |
| Degree and `coef0` | Polynomial kernel shape | Not relevant to the RBF kernel |
| Class/sample weights | Relative violation costs across rows | Must reflect a validated task tradeoff |
| Solver tolerance/iteration limit | Numerical optimization accuracy | Different from statistical regularization |

Weighting a class increases its effective violation penalty. It can change the boundary but does not guarantee desired recall or probability calibration. Tune decision thresholds separately on validation data if the default operating point is inappropriate.

## 13. Scores, Probabilities, and Multiclass Behavior

The natural SVM output is a decision score. Binary ranking metrics such as ROC-AUC can use an appropriately oriented score directly; they do not require probabilities. For a linear score, dividing by the coefficient norm gives signed distance in its feature coordinates. A nonlinear kernel score should not be described as ordinary Euclidean distance in the raw input space.

`SVC(probability=True)` fits an additional probability-estimation procedure. In the binary case this uses Platt scaling with internal cross-validation and adds training cost. Its predicted probability threshold can disagree with SVC's direct class prediction. Check probability quality independently rather than interpreting every large margin as a reliable probability. These implementation details are explained in the [SVM guide](https://scikit-learn.org/stable/modules/svm.html).

For strict validation of an entire calibrated pipeline, fit preprocessing within the calibration training folds too. Internal probability estimation in SVC receives the already transformed data from its enclosing pipeline; it does not independently refit that pipeline's scaler in its internal folds. An outer held-out test set still remains independent if the entire pipeline was fitted only on training data.

For $K$ classes, SVC fits one-versus-one binary models, a total of $K(K-1)/2$. Its default multiclass decision-score presentation does not change that internal strategy. Multiclass scores and probability columns require careful class-order handling; the binary sign rule should not be applied blindly to them.

## 14. Reading the Code Companion Carefully

`10_SVM.ipynb` loads `load_breast_cancer(as_frame=True)`, reserves 20% of rows using `random_state=42` and `stratify=y`, then fits two pipelines with `StandardScaler`:

| Model | Settings shown |
|---|---|
| Linear SVM | `SVC(kernel="linear", probability=True, random_state=42)` |
| RBF SVM | `SVC(kernel="rbf", C=1.0, gamma="scale", probability=True, random_state=42)` |

The companion compares accuracy, binary F1, and ROC-AUC. The dataset's label mapping is **0 = malignant, 1 = benign**, as documented in the [dataset reference](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_breast_cancer.html). Consequently, its default `f1_score(y_test, y_pred)` measures benign as the positive class, and its probability column `[:, 1]` scores benign for ROC-AUC. These are not automatically measures of malignant-case detection.

If malignant is the event of interest, define that positive class explicitly and align the chosen metric and probability/score orientation. For example, malignant recall concerns how many true label-0 rows were predicted as label 0. Do not infer that performance from the benign F1 alone.

The final loop prints `n_support_` and the total number of support vectors. Counts follow `classes_` order, while `support_vectors_` contains the feature coordinates seen by SVC—standardized values in these pipelines. Counts describe this fit, not a standalone quality score.

The companion does not perform hyperparameter tuning. These notes describe its source code without reporting measured scores, support-vector counts, or a winning kernel; it was not executed for this revision.

## 15. Evaluate the Chosen Operating Point

For a chosen positive class, a true positive (TP) is correctly identified, a false positive (FP) is incorrectly flagged positive, a false negative (FN) is a missed positive, and a true negative (TN) is correctly rejected.

$$
\mathrm{Accuracy}=\frac{TP+TN}{TP+TN+FP+FN},\qquad
\mathrm{Precision}=\frac{TP}{TP+FP},\qquad
\mathrm{Recall}=\frac{TP}{TP+FN},
$$

$$
F_1=\frac{2TP}{2TP+FP+FN}.
$$

Accuracy measures overall correctness, precision describes the positive predictions, and recall describes coverage of actual positives. F1 combines precision and recall while excluding true negatives. Zero denominators require an explicit evaluation convention.

For an illustrative defect detector with TP=18, FP=6, FN=2, and TN=74, accuracy is 92%, precision is 75%, recall is 90%, and F1 is $36/44\approx81.8\%$. These counts are a teaching example, not companion outputs.

ROC-AUC measures ranking across thresholds, while a confusion matrix evaluates actual class decisions at a particular operating point. Precision–recall analysis can be useful with rare positives. Calibration curves and probability scores assess another question: whether stated probabilities match observed frequencies.

<div style="color:#664b12; background-color:#fff5df; border-left:5px solid #b7791f; padding:10px 14px; margin:12px 0;"><strong>Evaluation caution</strong><br>Define the positive class and error costs before comparing scores. Strong overall accuracy can coexist with poor performance on the class that matters most.</div>

Select thresholds using development data and assess the final procedure on untouched test data. A margin, a probability, and a classification decision answer different questions.

## 16. Interpretation, Alternatives, and Computational Cost

For a linear SVM, a coefficient describes how changing a represented feature changes the score, with other represented features fixed. In standardized coordinates, a unit change is one training standard deviation. Correlated features can redistribute coefficients, so magnitude alone is not a definitive importance ranking. These are predictive associations, not causal effects.

An RBF SVM has no single raw-feature coefficient vector that explains its curved boundary. Support-vector locations can help diagnose the fit, but they do not provide a complete causal or feature-importance explanation.

| Alternative | Useful comparison |
|---|---|
| Logistic Regression | A linear conditional model with a different loss and direct probability link |
| LinearSVC | A scalable linear SVM implementation; defaults and optimization differ from linear-kernel SVC |
| Kernel SVC | Nonlinear flexibility when sample size and tuning budget permit |
| Naive Bayes | Fast distribution-based baseline, especially for suitable text representations |
| KNN | Local similarity classification with different storage and prediction costs |
| Trees or forests | Nonlinear interactions without the same scaling dependence |

SVC kernel training can become expensive as the number of rows grows. A full pairwise kernel matrix would have $n^2$ entries; implementations may use caching rather than storing all entries, but pairwise computations remain a major cost. Runtime also depends on separability, kernel settings, and solver convergence, so there is no single exact cost formula for every dataset.

For $s$ support vectors with $p$ dense features, an RBF prediction typically needs roughly $O(sp)$ work. A linear model represented by an explicit coefficient vector can score in roughly $O(p)$. Many support vectors can make a kernel model large and slow even if its feature dimension is modest.

## 17. Related Tasks, Limitations, and Recap

Support Vector Regression (SVR) adapts the idea to numerical targets. Its epsilon-insensitive loss is $\max(0,|y-f(\mathbf x)|-\epsilon)$, where $\epsilon\geq0$ specifies a tolerated error band. Errors inside that band incur no such loss; C controls the cost outside it. This is a regression objective, not the binary classification constraints derived earlier. One-class SVM is another related method for learning a region of typical observations.

| Failure mode | Why it matters | Practical response |
|---|---|---|
| Poor feature scales | Margins and kernels reflect unsuitable units | Fit appropriate preprocessing within folds |
| Mislabeled rows or extreme outliers | Violations can strongly affect the fit | Inspect data and validate C and weighting |
| Excessively local RBF fit | Training detail may not generalize | Tune C and gamma jointly |
| Too-simple representation | No useful boundary is available | Compare kernels, features, or other models |
| Very large training set | Kernel optimization becomes costly | Compare scalable linear or approximate methods |
| Distribution shift | A training margin does not protect against changed relationships | Evaluate realistic splits and monitor future performance |

SVM does not require normally distributed features, but it does depend on a useful representation and a validation design that matches future use. High dimensionality does not remove overfitting risk. A wide margin can be helpful without guaranteeing robustness to every perturbation or data shift.

The central idea is to learn a boundary that balances separation with margin violations. Support vectors carry the fitted kernel expansion; soft margins allow overlap; kernels change the geometry in which separation is learned. Choose scaling, C, and kernel settings through validation, and keep decision scores separate from calibrated probabilities.

Remember the worked example: a correctly classified point can lie inside the margin, and its score is not its probability. Understanding these distinctions makes both the mathematics and the companion's outputs easier to interpret.